# LLMs por dentro · Poner a prueba las slides

Las slides afirmaron cosas concretas:

| Slide | Afirmación |
|---|---|
| 6 | Un transformer produce una **distribución de probabilidad** sobre el siguiente token |
| 7–9 | Genera texto **en bucle**, token por token, hasta un token de fin o un máximo |
| 13 | Los tokens **no son palabras**, se parecen más a morfemas |
| 14 | Cada token es un **número entero**, su ID |
| 16 | Existe una **matriz de embeddings** $W_E$ con un vector por token |
| 19 | Palabras con significado parecido tienen **vectores parecidos** |
| 21 | Las relaciones se conservan: **rey − hombre + mujer ≈ reina** |
| 24–26 | El **producto punto** es positivo si dos vectores se parecen, cero si son ortogonales y negativo si se oponen |

En esta clase no vamos a creerle a ninguna. Vamos a comprobarlas una por una con un modelo
real corriendo en tu laptop, y al final vamos a usar todo eso para construir algo útil:
**un buscador que responde preguntas sobre el sílabo del curso**.

Todo corre local: **no hace falta API key ni tarjeta de crédito**.

## 0 · Preparación

```bash
uv sync --group hf --group llm
```

La primera ejecución descarga dos modelos (~1,5 GB en total). Después quedan en caché.

In [ ]:
import math
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tiktoken
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.manual_seed(0)
pd.set_option("display.max_colwidth", 120)

AZUL, NARANJA, GRIS, TINTA = "#1f6f8b", "#c1663f", "#9aa7b0", "#2b3a42"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA,
})

Usamos **Qwen2.5-0.5B-Instruct**: 494 millones de parámetros, entiende español y corre
en CPU. Es chico a propósito —un modelo de frontera tiene cientos de veces más
parámetros—, pero la maquinaria es exactamente la misma.

In [ ]:
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODELO)
llm = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).eval()

n_params = sum(p.numel() for p in llm.parameters())
print(f"{MODELO}  ·  {n_params / 1e6:,.0f} millones de parámetros")

## 1 · Tokens

### 1.1 · La slide 14, reproducida

La slide mostró un texto y su lista de token IDs, citando «el ejemplo de OpenAI». Si la
slide es correcta, deberíamos poder obtener **exactamente** los mismos números.

In [ ]:
texto_slide = (
    "Transformers first tokenize input words. Then they create embeddings from the tokens. "
    "And, finally, they recursively predict the next token until some stopping criterion is met."
)
ids_slide = [12200, 409, 1577, 192720, 3422, 6391, 13, 7801, 1023, 2501, 174989, 591, 290,
             20290, 13, 1958, 11, 8486, 11, 1023, 130266, 17946, 290, 2613, 6602, 4609, 1236,
             36616, 71270, 382, 1421, 13]

for nombre in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(nombre)
    ids = enc.encode(texto_slide)
    print(f"{nombre:<12}  {len(ids)} tokens  ·  ¿coincide con la slide? {ids == ids_slide}")

Coincidencia exacta con `o200k_base`, el tokenizador de GPT-4o. Ahora sabemos qué usó
la slide. `cl100k_base` (GPT-4) produce la misma cantidad de tokens pero **otros números**:
el ID de un token solo tiene sentido dentro de su propio vocabulario.

Veamos qué hay detrás de cada número.

In [ ]:
enc = tiktoken.get_encoding("o200k_base")

def ver_tokens(texto, encoder=enc):
    """Muestra el texto partido en tokens, separados por │"""
    ids = encoder.encode(texto)
    piezas = [encoder.decode([i]) for i in ids]
    print("│".join(piezas))
    print(f"→ {len(ids)} tokens")
    return piezas

_ = ver_tokens(texto_slide)

### 1.2 · «Una mentira conveniente» (slide 13)

La slide advirtió que tratar tokens como palabras es una simplificación. Con palabras
peruanas se ve de inmediato:

In [ ]:
palabras = ["Lima", "ceviche", "Huancavelica", "ferrocarril", "desafortunadamente",
            "anticonstitucionalmente", "inflación", "BCRP"]

pd.DataFrame({
    "palabra": palabras,
    "tokens": [[enc.decode([i]) for i in enc.encode(p)] for p in palabras],
    "n": [len(enc.encode(p)) for p in palabras],
})

«desafortunadamente» → `des` + `af` + `ortun` + `adamente`: el tokenizador aprendió
pedazos frecuentes, y algunos coinciden con morfemas (`-mente`). **Ninguna** de estas
palabras es un solo token, ni siquiera «Lima».

### 1.3 · Consecuencia práctica: ¿por qué un LLM no sabe contar letras?

Si el modelo nunca ve letras sino tokens, una pregunta trivial se vuelve difícil.

In [ ]:
palabra = "ferrocarril"
print("lo que ves     :", " ".join(palabra))
print("lo que ve el LLM:", [tok.decode([i]) for i in tok.encode(palabra)])
print(f"respuesta real : {palabra.count('r')} letras 'r'")

In [ ]:
def chat(mensaje, sistema=None, max_tokens=60):
    """Una respuesta del modelo, sin azar (greedy)."""
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + \
           [{"role": "user", "content": mensaje}]
    entrada = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
    with torch.no_grad():
        salida = llm.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chat("¿Cuántas letras r tiene la palabra ferrocarril? Responde solo con un número."))

En nuestra corrida respondió **3**; la respuesta correcta es 4. El modelo nunca «vio» las
cuatro erres: vio `fer`, `roc`, `arr` e `il`.

### 1.4 · Consecuencia práctica: escribir en español cuesta más

Las APIs cobran **por token**, y el tamaño máximo de un documento también se mide en
tokens. Mismo contenido, dos idiomas:

In [ ]:
texto_es = (
    "Los transformers primero tokenizan las palabras de entrada. Luego crean embeddings a partir "
    "de los tokens. Y, finalmente, predicen recursivamente el siguiente token hasta que se cumple "
    "algún criterio de parada."
)

n_en = len(enc.encode(texto_slide))
n_es = len(enc.encode(texto_es))
print(f"inglés : {n_en} tokens")
print(f"español: {n_es} tokens   →  {n_es / n_en - 1:+.0%}")

In [ ]:
# Supuesto ilustrativo: ajusta el precio al del proveedor que uses
PRECIO_USD_POR_MILLON = 3.00
TOKENS_POR_RECLAMO_EN = 400          # un reclamo de consumidor típico, en inglés
N_RECLAMOS = 500_000

for idioma, factor in [("inglés", 1.0), ("español", n_es / n_en)]:
    tokens = N_RECLAMOS * TOKENS_POR_RECLAMO_EN * factor
    print(f"{idioma:<8} {tokens / 1e6:6,.0f} M tokens  →  US$ {tokens / 1e6 * PRECIO_USD_POR_MILLON:8,.0f}")

Procesar el mismo volumen de texto en español sale más caro, solo por cómo se trocea el
idioma. Cuando dimensiones un proyecto con LLMs, **cuenta tokens, no palabras**.

### 1.5 · Cada modelo tiene su propio vocabulario

El tokenizador de nuestro Qwen no es el de GPT-4o:

In [ ]:
frase = "El Banco Central de Reserva del Perú mantuvo la tasa de referencia."

for nombre, ids in [("GPT-4o (o200k_base)", enc.encode(frase)),
                    ("Qwen2.5", tok.encode(frase))]:
    print(f"{nombre:<22} {len(ids):>2} tokens   primeros IDs: {ids[:6]}")

## 2 · Predecir el siguiente token

### 2.1 · La distribución de probabilidad (slide 6)

La slide dijo que el modelo **no produce una palabra**: produce una probabilidad para
**cada uno** de los tokens de su vocabulario. Pidámosla directamente.

In [ ]:
def siguiente(prompt, k=10):
    """Probabilidad de cada token posible después del prompt."""
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = llm(ids).logits[0, -1]          # un puntaje por token del vocabulario
    probs = torch.softmax(logits, dim=-1)          # puntajes -> probabilidades
    top = torch.topk(probs, k)
    tabla = pd.DataFrame({
        "token": [repr(tok.decode([i])) for i in top.indices],
        "prob": top.values.numpy(),
    })
    return tabla, probs

tabla, probs = siguiente("La capital del Perú es")
print(f"tokens en el vocabulario : {probs.numel():,}")
print(f"suma de probabilidades   : {probs.sum():.4f}")
tabla

Tres cosas para mirar con calma:

1. **Es una distribución de verdad**: 151 936 probabilidades que suman 1.
2. **«Lima» no está arriba.** El modelo no «sabe la respuesta» y la escupe; predice qué
   token suele venir después. Después de «es», lo más común en español es «la», «una», «un».
3. Ninguna opción pasa del 17 %: el modelo está **muy inseguro**.

¿Qué pasa si le damos más contexto?

In [ ]:
prompts = ["La capital del Perú es", "La capital del Perú es la ciudad de"]
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)

for eje, prompt in zip(ejes, prompts):
    t, _ = siguiente(prompt, k=8)
    t = t.iloc[::-1]
    colores = [NARANJA if "Lima" in x else AZUL for x in t["token"]]
    eje.barh(t["token"], t["prob"], color=colores, height=0.65)
    for y, p in enumerate(t["prob"]):
        eje.text(p + 0.005, y, f"{p:.0%}", va="center", fontsize=8, color=TINTA)
    eje.set_title(f"«{prompt} ___»", fontsize=9.5, color=TINTA)
    eje.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    eje.grid(axis="x", alpha=0.25)

fig.suptitle("El contexto concentra la probabilidad", fontsize=11, color=TINTA)
plt.tight_layout()
plt.show()

Con cuatro palabras más, «Lima» salta al primer lugar y casi duplica su probabilidad.

### 2.2 · ¿Cuándo está seguro el modelo?

In [ ]:
casos = [
    "El Banco Central de Reserva del Perú fija la tasa de",
    "El color que más me gusta es el",
    "La capital del Perú es la ciudad de",
    "El presidente del Perú en 2030 será",
    "Mi nombre es",
]

filas = []
for p in casos:
    t, _ = siguiente(p, k=3)
    filas.append({"prompt": p,
                  "más probable": f"{t.iloc[0]['token']} ({t.iloc[0]['prob']:.0%})",
                  "2.º": f"{t.iloc[1]['token']} ({t.iloc[1]['prob']:.0%})",
                  "3.º": f"{t.iloc[2]['token']} ({t.iloc[2]['prob']:.0%})"})
pd.DataFrame(filas)

Cuando el contexto casi determina la continuación («fija la tasa de» → «interés», 59 %), la
probabilidad se concentra. Cuando hay miles de continuaciones válidas («Mi nombre es» →
«Carlos», apenas 5 %), se reparte. **Una distribución plana no es un error: es el modelo
reflejando que la pregunta tiene muchas respuestas.**

### 2.3 · El bucle, escrito a mano (slides 7 y 9)

La slide dijo que la generación es **recursiva**: el token elegido se agrega al texto y
se vuelve a predecir, hasta encontrar un token de fin o llegar a un máximo. Son diez
líneas de código:

In [ ]:
def generar_a_mano(prompt, max_tokens=12):
    ids = tok(prompt, return_tensors="pt").input_ids
    pasos = []
    for paso in range(max_tokens):
        with torch.no_grad():
            probs = torch.softmax(llm(ids).logits[0, -1], dim=-1)
        nuevo = int(torch.argmax(probs))                       # el más probable (greedy)
        pasos.append({"paso": paso + 1, "token": repr(tok.decode([nuevo])),
                      "prob": f"{probs[nuevo]:.0%}"})
        if nuevo == tok.eos_token_id:                          # criterio de parada 1
            break
        ids = torch.cat([ids, torch.tensor([[nuevo]])], dim=1)  # la salida vuelve a entrar
    return tok.decode(ids[0]), pd.DataFrame(pasos)             # criterio de parada 2: max_tokens

texto, pasos = generar_a_mano("El Banco Central de Reserva del Perú fija la tasa de")
print(texto)
pasos

Comprobemos que nuestro bucle hace lo mismo que la función oficial `generate`:

In [ ]:
prompt = "El Banco Central de Reserva del Perú fija la tasa de"
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    oficial = llm.generate(ids, max_new_tokens=12, do_sample=False)

print("a mano  :", texto)
print("generate:", tok.decode(oficial[0]))
print("¿idénticos?", texto == tok.decode(oficial[0]))

**No coinciden**, y la razón es instructiva. El fabricante trae valores por defecto que
`generate` aplica aunque no los pidas:

In [ ]:
cfg = llm.generation_config
print(f"repetition_penalty = {cfg.repetition_penalty}   (castiga repetir tokens ya usados)")
print(f"temperature        = {cfg.temperature}")
print(f"top_p              = {cfg.top_p}")
print(f"top_k              = {cfg.top_k}")

Aunque pedimos `do_sample=False`, la penalización por repetición sigue modificando las
probabilidades. Nuestro bucle no la aplica. Si la apagamos, la mecánica es idéntica:

In [ ]:
with torch.no_grad():
    oficial = llm.generate(ids, max_new_tokens=12, do_sample=False, repetition_penalty=1.0)

print("a mano  :", texto)
print("generate:", tok.decode(oficial[0]))
print("¿idénticos?", texto == tok.decode(oficial[0]))

Lección práctica: **dos personas con el mismo modelo y el mismo prompt pueden obtener
respuestas distintas** solo porque una usa los valores por defecto y la otra no. Cuando
reportes resultados de un LLM, reporta también su configuración.

### 2.4 · Temperatura: por qué ChatGPT responde distinto cada vez

La slide 6 dijo que el modelo **muestrea** un token de la distribución. Hasta ahora
elegimos siempre el más probable. La **temperatura** decide cuánto se arriesga:

$$p_i(T) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Temperatura baja afila la distribución; alta la aplana.

In [ ]:
prompt = "Un buen nombre para una cevichería en Miraflores es"
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    logits = llm(ids).logits[0, -1]

temperaturas = [0.3, 1.0, 2.0]
fig, ejes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
top_ids = torch.topk(logits, 10).indices
etiquetas = [repr(tok.decode([i])) for i in top_ids]

for eje, T in zip(ejes, temperaturas):
    p = torch.softmax(logits / T, dim=-1)[top_ids].numpy()
    eje.bar(range(10), p, color=AZUL, width=0.7)
    eje.set_xticks(range(10), etiquetas, rotation=60, ha="right", fontsize=7.5)
    eje.set_title(f"T = {T}   ·   estos 10 tokens suman {p.sum():.0%}", fontsize=9.5, color=TINTA)
    eje.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    eje.grid(axis="y", alpha=0.25)

ejes[0].set_ylabel("probabilidad")
fig.suptitle(f"«{prompt} ___»", fontsize=10.5, color=TINTA)
plt.tight_layout()
plt.show()

Con T = 0.3 un solo token se lleva casi toda la probabilidad. Con T = 2.0 el panel parece
vacío, y no es un error: los 10 tokens más probables apenas suman unos puntos porcentuales,
porque el resto se reparte entre los otros 151 926 tokens del vocabulario.

In [ ]:
def muestrear(prompt, T, n=4, max_tokens=12):
    ids = tok(prompt, return_tensors="pt").input_ids
    torch.manual_seed(42)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=max_tokens, do_sample=True, temperature=T,
                           top_k=0, top_p=1.0, repetition_penalty=1.0,   # solo temperatura
                           num_return_sequences=n, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[ids.shape[1]:], skip_special_tokens=True).strip().split("\n")[0]
            for o in out]

for T in temperaturas:
    print(f"\nT = {T}")
    for s in muestrear(prompt, T):
        print("   ·", s)

Con T = 0.3 las respuestas se repiten; con T = 2.0 aparecen combinaciones que ningún
humano escribiría. **Regla práctica:**

| Tarea | Temperatura |
|---|---|
| Extraer datos, clasificar, responder sobre documentos | 0 (o casi) |
| Redactar, resumir con variedad | ~0.7 |
| Lluvia de ideas | ~1.0 |

## 3 · La matriz de embeddings $W_E$ (slides 16–17)

La slide dijo que dentro del modelo hay una matriz con **un vector por token**, cuyas
entradas se aprenden con datos. No es una metáfora: la podemos sacar del modelo.

In [ ]:
W_E = llm.get_input_embeddings().weight.detach()

vocab, dim = W_E.shape
print(f"W_E              : {vocab:,} tokens × {dim} dimensiones")
print(f"parámetros en W_E: {W_E.numel() / 1e6:,.0f} M  ({W_E.numel() / n_params:.0%} de todo el modelo)")

Más de un cuarto del modelo es **solo la tabla de vectores**. Un detalle de notación: la
slide pone cada palabra en una **columna**; PyTorch la guarda en una **fila**. Misma idea,
matriz traspuesta.

El vector de un token es simplemente su fila:

In [ ]:
id_lima = tok.encode(" Lima")
print("tokens de ' Lima':", id_lima, [tok.decode([i]) for i in id_lima])

vec = W_E[id_lima[0]]
print(f"E(' Lima') = [{', '.join(f'{x:+.3f}' for x in vec[:8])}, ...]  ({len(vec)} números)")

## 4 · El producto punto (slides 23–26)

### 4.1 · Positivo, cero, negativo

$$v \cdot w = \sum_{i=1}^{n} v_i\, w_i$$

In [ ]:
v = np.array([3.0, 1.0])
casos = {
    "parecidos":  np.array([2.5, 1.5]),
    "ortogonales": np.array([-1.0, 3.0]),
    "opuestos":   np.array([-3.0, -0.8]),
}

fig, ejes = plt.subplots(1, 3, figsize=(10, 3.3))
for eje, (nombre, w) in zip(ejes, casos.items()):
    for vec, color, etiqueta in [(v, AZUL, "v"), (w, NARANJA, "w")]:
        eje.annotate("", xy=vec, xytext=(0, 0),
                     arrowprops=dict(arrowstyle="-|>", color=color, lw=2))
        eje.text(*(vec * 1.12), etiqueta, color=color, fontsize=10, ha="center", va="center")
    eje.set_xlim(-3.8, 3.8); eje.set_ylim(-2.5, 3.8); eje.set_aspect("equal")
    eje.axhline(0, color=GRIS, lw=0.6); eje.axvline(0, color=GRIS, lw=0.6)
    eje.set_xticks([]); eje.set_yticks([])
    eje.set_title(f"{nombre}\nv · w = {v @ w:+.1f}", fontsize=10, color=TINTA)

plt.tight_layout()
plt.show()

### 4.2 · La trampa de la magnitud: por qué se usa similitud coseno

El producto punto también crece con el **largo** del vector, no solo con la dirección.

In [ ]:
consulta   = np.array([1.0, 1.0])
parecido   = np.array([1.0, 0.9])     # misma dirección, largo normal
distinto_largo = np.array([6.0, 0.5])  # otra dirección, pero muy largo

def coseno(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

pd.DataFrame({
    "producto punto": [consulta @ parecido, consulta @ distinto_largo],
    "similitud coseno": [coseno(consulta, parecido), coseno(consulta, distinto_largo)],
}, index=["parecido", "distinto pero largo"]).round(3)

Con producto punto crudo gana el vector largo aunque apunte a otro lado. **Normalizando**
(dividiendo por los largos) queda solo la dirección: eso es la similitud coseno, y es lo que
usan los buscadores semánticos. Si los vectores ya tienen largo 1, producto punto y coseno
son lo mismo.

### 4.3 · Significado = cercanía, en la $W_E$ real (slide 19)

Buscamos los tokens cuyo vector está más cerca de otro. Filtramos a tokens que son
palabras completas para que la lista sea legible.

In [ ]:
# Tokens "limpios": empiezan con espacio (inicio de palabra) y son solo letras
piezas = [tok.decode([i]) for i in range(vocab)]
es_palabra = np.array([
    p.startswith(" ") and re.fullmatch(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]{3,}", p.strip()) is not None
    for p in piezas
])
W_norm = torch.nn.functional.normalize(W_E, dim=1)
print(f"tokens que son palabras completas: {es_palabra.sum():,} de {vocab:,}")

def id_unico(palabra):
    ids = tok.encode(" " + palabra)
    return ids[0] if len(ids) == 1 else None

def vecinos(vector, excluir=(), k=6):
    sims = (W_norm @ torch.nn.functional.normalize(vector, dim=0)).numpy()
    sims[~es_palabra] = -1
    vistos, out = {e.lower() for e in excluir}, []
    for i in np.argsort(-sims):
        w = piezas[i].strip()
        if w.lower() not in vistos:
            vistos.add(w.lower()); out.append(f"{w} ({sims[i]:.2f})")
        if len(out) == k:
            break
    return out

for palabra in ["Peru", "inflation", "football", "doctor"]:
    i = id_unico(palabra)
    print(f"{palabra:<10} →", ", ".join(vecinos(W_E[i], excluir=[palabra])))

«Peru» queda rodeado de países vecinos y de Lima; «football», de otros deportes; «doctor»,
de profesiones médicas —incluso en español, «médico»—. La slide 19 se cumple.

Pero mira «inflation»: sus vecinos más cercanos son «inflated» e «inflatable», que no
tienen nada que ver con precios. En $W_E$ se mezclan **significado y forma de la palabra**.

## 5 · Relaciones vectoriales (slides 20–22)

La slide afirmó que $E(\text{king}) - E(\text{man}) + E(\text{woman}) \approx E(\text{queen})$.
Esa figura clásica viene de modelos como *word2vec*. ¿Se cumple en la $W_E$ de un LLM
moderno?

In [ ]:
def analogia(a, b, c, k=5):
    """a - b + c ≈ ?"""
    ids = {p: id_unico(p) for p in (a, b, c)}
    partidas = [p for p, i in ids.items() if i is None]
    if partidas:
        return f"no se puede: {partidas} no es un solo token"
    vector = W_E[ids[a]] - W_E[ids[b]] + W_E[ids[c]]
    return ", ".join(vecinos(vector, excluir=[a, b, c], k=k))

pruebas = [
    ("king", "man", "woman"),
    ("queen", "woman", "man"),
    ("walking", "walk", "swim"),
    ("bigger", "big", "small"),
    ("Paris", "France", "Peru"),
    ("Tokyo", "Japan", "Peru"),
]
for a, b, c in pruebas:
    print(f"{a} − {b} + {c:<6} →  {analogia(a, b, c)}")

**Se cumple.** «queen» sale primero, las relaciones gramaticales (−ing, −er) también
funcionan, y la analogía capital–país lleva de Tokio a **Lima**. La slide tenía razón,
incluso en un modelo pequeño.

### 5.1 · Donde se rompe

La slide 22 mostró comidas típicas por nacionalidad. Probemos la versión peruana:

In [ ]:
comidas = ["pizza", "sushi", "tacos", "pasta", "curry", "burger", "noodles",
           "ceviche", "paella", "ramen", "empanadas", "arepa"]

enteras = [c for c in comidas if id_unico(c) is not None]
partidas = {c: [tok.decode([i]) for i in tok.encode(" " + c)] for c in comidas if id_unico(c) is None}
print("un solo token :", enteras)
print("partidas      :", partidas)

Solo 7 de las 12 comidas existen como token. Hagamos la analogía obligando al modelo a
elegir **entre las comidas que sí puede representar**:

In [ ]:
def analogia_entre(a, b, c, candidatas):
    vector = torch.nn.functional.normalize(W_E[id_unico(a)] - W_E[id_unico(b)] + W_E[id_unico(c)], dim=0)
    puntajes = {x: float(W_norm[id_unico(x)] @ vector) for x in candidatas if x != a}
    return ", ".join(f"{x} ({s:.2f})" for x, s in sorted(puntajes.items(), key=lambda kv: -kv[1])[:4])

print("sushi − Japan + Peru  →", analogia_entre("sushi", "Japan", "Peru", enteras))
print("pasta − Italy + India →", analogia_entre("pasta", "Italy", "India", enteras))
print()
print("rey − hombre + mujer  →", analogia("rey", "hombre", "mujer"))

Aquí se juntan las dos mitades de la clase:

- El vector **sí se mueve** hacia «comida latinoamericana»: elige «tacos». Y de la pasta
  italiana a la India llega a «noodles» y «curry». La dirección existe.
- Pero **«ceviche» no puede ganar porque no es un token**: se parte en `ce` + `v` + `iche`.
  La «mentira conveniente» de la slide 13 tiene un costo real.
- **«rey» tampoco es un token.** El español se fragmenta más (lo vimos en 1.4), y eso
  degrada estas operaciones a nivel de token.

La lección no es que las slides mientan, sino que el significado en un LLM **no vive solo en
$W_E$**: se construye capa por capa. Para trabajar con significado de frases enteras se usan
modelos entrenados específicamente para eso. Es lo que haremos ahora.

## 6 · Aplicación: pregúntale al sílabo

Juntemos todo. Embeddings + producto punto = **búsqueda por significado**. Es la «R» de
**RAG** (*Retrieval-Augmented Generation*): antes de que el LLM responda, buscamos los
fragmentos de un documento que se parecen a la pregunta y se los damos como contexto.

El documento: el sílabo de este curso.

### 6.1 · Trocear el documento

In [ ]:
def raiz_repo():
    p = Path.cwd()
    while not (p / "pyproject.toml").exists() and p != p.parent:
        p = p.parent
    return p

ACENTOS = {r"\'a": "á", r"\'e": "é", r"\'i": "í", r"\'\i": "í", r"\'o": "ó", r"\'u": "ú",
           r"\'A": "Á", r"\'E": "É", r"\'O": "Ó", r"\'U": "Ú", r"\~n": "ñ"}

def limpiar_latex(s):
    for k, v in ACENTOS.items():
        s = s.replace(k, v)
    s = re.sub(r"\\href\{[^}]*\}\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\url\{[^}]*\}", "", s)
    s = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "", s)                      # sin correos
    for _ in range(2):
        s = re.sub(r"\\(textbf|emph|textit|texttt)\{([^{}]*)\}", r"\2", s)
    s = s.replace(r"\,", " ").replace(r"\%", "%").replace("---", "—").replace("--", "–")
    s = re.sub(r"\$\\pm\s*(\d+)\$", r"±\1", s)
    s = re.sub(r"\\item\s*", "", s)
    s = re.sub(r"\\[a-zA-Z]+\*?(\[[^\]]*\])?(\{[^}]*\})?", "", s)
    s = s.replace("{", "").replace("}", "")
    return re.sub(r"\s+", " ", s).strip(" ,&")

def trocear_silabo(ruta):
    """Un fragmento por párrafo, ítem o fila de tabla."""
    cuerpo = Path(ruta).read_text(encoding="utf-8").split(r"\begin{document}", 1)[-1]
    partes = re.split(r"\\section\{([^}]*)\}", cuerpo)
    out = []
    for titulo, bloque in zip(partes[1::2], partes[2::2]):
        titulo = limpiar_latex(titulo)
        if titulo.startswith("Bibliograf"):
            continue
        sub, encabezado, parrafo = "", [], []

        def guardar(texto):
            if len(texto) > 12:
                out.append({"seccion": titulo + (f" › {sub}" if sub else ""), "texto": texto})

        def cerrar():
            if parrafo:
                t = limpiar_latex(" ".join(parrafo))
                if len(t) > 25:
                    guardar(t)
                parrafo.clear()

        for linea in bloque.split("\n"):
            l = linea.strip()
            if l.startswith("%"):
                continue
            m = re.match(r"\\subsection\*?\{(.*)\}", l)
            if m:
                cerrar(); sub = limpiar_latex(m.group(1)); continue
            if "&" in l and l.endswith("\\\\"):                          # fila de tabla
                cerrar()
                celdas = [limpiar_latex(c) for c in l.rstrip("\\").split("&")]
                if celdas[0] in ("Rubro", "Sem."):
                    encabezado = celdas; continue
                if celdas[0] == "Total" or not any(celdas):
                    continue
                if encabezado and len(encabezado) == len(celdas):
                    guardar("; ".join(f"{h}: {c}" for h, c in zip(encabezado, celdas) if c))
                else:
                    guardar(" · ".join(c for c in celdas if c))
                continue
            if l.startswith(r"\item"):
                cerrar(); parrafo.append(l); cerrar(); continue
            if not l or l.startswith(("\\begin", "\\end")) or l in (
                    r"\toprule", r"\midrule", r"\bottomrule",
                    r"\endfirsthead", r"\endhead", r"\endlastfoot"):
                cerrar(); continue
            parrafo.append(l)
        cerrar()
    return out

fragmentos = trocear_silabo(raiz_repo() / "syllabus" / "Silabo_DS_Python_UP_2026_II.tex")
print(f"{len(fragmentos)} fragmentos")
pd.DataFrame(fragmentos).groupby("seccion").size().rename("fragmentos").to_frame()

Una decisión de troceo que parece menor y no lo es: **las filas de tabla llevan sus
encabezados**. Sin eso, la fila del trabajo final quedaría así:

In [ ]:
fila = next(f["texto"] for f in fragmentos if "Peso (%): 30" in f["texto"])
print("sin encabezados:  Trabajo final · Proyecto de startup desplegada, con pitch y sustentación · 30")
print("con encabezados: ", fila)

Un «30» suelto no le dice nada a nadie —ni a un buscador ni a un LLM—. «Peso (%): 30» sí.
Al trocear tablas, cada fila tiene que poder leerse sola.

### 6.2 · Embeddings de frases

Usamos `multilingual-e5-small`, un modelo entrenado para que frases con el mismo
significado tengan vectores cercanos. Convierte cada fragmento en un vector de 384 números
**de largo 1**, así que el producto punto ya es la similitud coseno.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-small")

docs = [f"{f['seccion']}: {f['texto']}" for f in fragmentos]
# e5 fue entrenado con prefijos: "passage:" para documentos, "query:" para preguntas
E = embedder.encode([f"passage: {d}" for d in docs], normalize_embeddings=True)

print(f"matriz de embeddings: {E.shape}")
print(f"largo del primer vector: {np.linalg.norm(E[0]):.4f}")

Antes de buscar, una comprobación visual de la slide 19 a nivel de frases: textos de
temas distintos deberían formar bloques.

In [ ]:
frases = [
    "El BCRP subió la tasa de interés de referencia.",
    "La inflación en Lima cerró el año por debajo del 3 %.",
    "Alianza Lima ganó el clásico en Matute.",
    "La selección peruana perdió en las eliminatorias.",
    "El ceviche lleva limón, ají y pescado fresco.",
    "El lomo saltado se sirve con papas fritas y arroz.",
]
V = embedder.encode([f"query: {f}" for f in frases], normalize_embeddings=True)
S = V @ V.T                                     # todos los productos punto a la vez

fig, eje = plt.subplots(figsize=(6.4, 5.2))
im = eje.imshow(S, cmap="Blues", vmin=S[~np.eye(len(S), dtype=bool)].min(), vmax=1)
cortas = ["BCRP tasa", "inflación", "Alianza", "selección", "ceviche", "lomo saltado"]
eje.set_xticks(range(6), cortas, rotation=35, ha="right")
eje.set_yticks(range(6), cortas)
for i in range(6):
    for j in range(6):
        eje.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if S[i, j] > 0.9 else TINTA)
eje.set_title("Similitud coseno entre frases\n(economía · fútbol · comida)", fontsize=10, color=TINTA)
eje.grid(False)
plt.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

Aparecen tres bloques: economía, fútbol y comida se parecen más entre sí. Fíjate en la
escala: en este modelo **todas** las similitudes están entre 0.81 y 0.88.
**Lo que importa es el orden, no el valor absoluto**, y los umbrales («mayor a 0.8 es
relevante») no se trasladan de un modelo a otro.

### 6.3 · Buscar

In [ ]:
def buscar(pregunta, k=3):
    q = embedder.encode([f"query: {pregunta}"], normalize_embeddings=True)[0]
    sims = E @ q                                    # producto punto contra los 74 fragmentos
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"similitud": sims[top].round(3),
                         "fragmento": [docs[i][:115] for i in top]})

buscar("¿Puedo hacer el proyecto final con mis amigos?")

La pregunta habla de «amigos»; el sílabo habla de «individual», «solo founder» y «sin
equipos». La idea clave no comparte palabras, y aun así el fragmento correcto sale primero.

Ahora cambiemos la pregunta apenas un poco:

In [ ]:
def puesto(pregunta, clave):
    """En qué lugar del ranking queda el fragmento que contiene `clave`."""
    q = embedder.encode([f"query: {pregunta}"], normalize_embeddings=True)[0]
    orden = np.argsort(-(E @ q))
    correcto = next(i for i, d in enumerate(docs) if clave in d)
    return int(np.where(orden == correcto)[0][0]) + 1

for p in ["¿Puedo hacer el proyecto final con mis amigos?",
          "¿Me puedo juntar con mis amigos para el proyecto final?",
          "¿Me puedo juntar con mis amigos para el proyecto?"]:
    print(f"puesto {puesto(p, 'solo founder'):>2}   {p}")

Quitar la palabra «final» manda el fragmento correcto del puesto 1 al 20.

### 6.4 · Medir antes de confiar

Un buscador que funciona con un ejemplo no demuestra nada. Armamos 10 preguntas, cada una
escrita de dos formas —con las palabras del sílabo y con las de un alumno— y marcamos
cuál es el fragmento correcto.

In [ ]:
evaluacion = pd.DataFrame([
 ("¿Cuánto pesa el trabajo final en la nota?",          "¿Qué porcentaje de mi calificación es el proyecto de fin de ciclo?",   "Peso (%): 30"),
 ("¿El trabajo final es individual o en equipos?",       "¿Me puedo juntar con mis amigos para el proyecto?",                    "solo founder"),
 ("¿Se elimina la nota más baja de las prácticas?",      "¿Qué pasa si me va mal en una de las evaluaciones cortas?",            "Se elimina la nota más baja"),
 ("¿Cuántos minutos dura el pitch?",                     "¿Cuánto tiempo tengo para presentar mi startup frente al jurado?",     "Pitch de 7 minutos"),
 ("¿Se puede usar Claude Code o Codex?",                 "¿Está permitido que la inteligencia artificial me ayude a programar?", "Se puede y se debe usar"),
 ("¿Qué día es la clase de teoría?",                     "¿A qué hora tengo que llegar el martes?",                               "Horas Teoría"),
 ("¿Cuándo vemos LLMs y RAG?",                           "¿En qué fecha toca la clase de modelos de lenguaje?",                  "Sem.: 8;"),
 ("¿Cuándo es la semana de exámenes parciales?",         "¿Qué semana no hay clases por los parciales?",                         "Semana de exámenes parciales — sin clases"),
 ("¿Cuándo es la devolución de calificaciones finales?", "¿Qué día publican las notas del curso?",                               "Devolución de calificaciones"),
 ("¿Qué incluye el análisis geoespacial?",               "¿Vamos a aprender a hacer mapas?",                                      "6. Análisis geoespacial"),
], columns=["literal", "parafraseo", "clave"])

evaluacion["correcto"] = [next(i for i, d in enumerate(docs) if c in d) for c in evaluacion["clave"]]
evaluacion[["literal", "parafraseo"]]

Comparamos contra un buscador **por palabras** (TF-IDF, lo que haría un Ctrl+F sofisticado).
La métrica es *hit@k*: ¿el fragmento correcto está entre los primeros *k* resultados?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(strip_accents="unicode").fit(docs)
D_tfidf = tfidf.transform(docs)

buscadores = {
    "palabras (TF-IDF)": lambda q: (D_tfidf @ tfidf.transform([q]).T).toarray().ravel(),
    "embeddings (e5)":   lambda q: E @ embedder.encode([f"query: {q}"], normalize_embeddings=True)[0],
}

filas = []
for nombre, puntuar in buscadores.items():
    for tipo in ["literal", "parafraseo"]:
        rangos = [int(np.where(np.argsort(-puntuar(q)) == c)[0][0]) + 1
                  for q, c in zip(evaluacion[tipo], evaluacion["correcto"])]
        filas.append({"buscador": nombre, "pregunta": tipo,
                      "hit@1": sum(r <= 1 for r in rangos), "hit@3": sum(r <= 3 for r in rangos)})

resultados = pd.DataFrame(filas)
resultados

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
colores = {"palabras (TF-IDF)": GRIS, "embeddings (e5)": AZUL}
x = np.arange(2)

for eje, metrica in zip(ejes, ["hit@1", "hit@3"]):
    for j, (nombre, color) in enumerate(colores.items()):
        vals = resultados.query("buscador == @nombre")[metrica].values
        barras = eje.bar(x + (j - 0.5) * 0.36, vals, width=0.34, color=color, label=nombre)
        for b, v in zip(barras, vals):
            eje.text(b.get_x() + b.get_width() / 2, v + 0.2, f"{v}/10",
                     ha="center", fontsize=8.5, color=TINTA)
    eje.set_xticks(x, ["con palabras del sílabo", "con palabras del alumno"])
    eje.set_ylim(0, 11)
    eje.set_title(metrica, fontsize=10, color=TINTA)
    eje.grid(axis="y", alpha=0.25)

ejes[0].set_ylabel("preguntas acertadas (de 10)")
ejes[0].legend(frameon=False, fontsize=8.5, loc="upper right")
fig.suptitle("¿Qué buscador encuentra el fragmento correcto?", fontsize=11, color=TINTA)
plt.tight_layout()
plt.show()

Cuando la pregunta usa las palabras del documento, los dos buscadores andan parecido.
Cuando el alumno pregunta **con sus propias palabras**, el buscador por palabras se
desploma y el de embeddings aguanta.

Y ninguno es perfecto en *hit@1*. Por eso RAG no le pasa al LLM **un** fragmento sino
**varios** (aquí, 3): la probabilidad de que el correcto esté entre ellos es mucho mayor.

### 6.5 · RAG en miniatura

Última pieza: le damos al LLM la pregunta **con** y **sin** los fragmentos recuperados.

In [ ]:
INSTRUCCION = "Responde la pregunta usando solo la información dada. Responde en español con una oración completa."

def rag(pregunta, k=3):
    q = embedder.encode([f"query: {pregunta}"], normalize_embeddings=True)[0]
    top = np.argsort(-(E @ q))[:k]
    contexto = "\n\n".join(fragmentos[i]["texto"] for i in top)
    respuesta = chat(f"Texto:\n{contexto}\n\nPregunta: {pregunta}", sistema=INSTRUCCION)
    return respuesta, [fragmentos[i]["texto"][:90] for i in top]

preguntas = [
    "¿Me puedo juntar con mis amigos para el proyecto final?",
    "¿Cuánto tiempo dura el pitch?",
    "¿Qué día publican las notas finales?",
    "¿Puedo usar Claude Code para programar?",
    "¿Qué porcentaje de la nota es el trabajo final?",
    "¿Cuándo vemos la clase de LLMs?",
]

for p in preguntas:
    con, recuperado = rag(p)
    print(f"■ {p}")
    print(f"   SIN contexto: {chat(p).replace(chr(10), ' ')[:150]}")
    print(f"   CON contexto: {con}")
    print(f"   recuperó    : {recuperado[0]}…")
    print()

Mira las respuestas **sin contexto**: el modelo no duda, no dice «no sé», y lo que dice es
inventado. Eso es una **alucinación**, y sale del mecanismo de la sección 2 —el modelo
siempre produce el token más probable, sepa o no la respuesta—.

Con contexto, cuatro respuestas pasan a estar ancladas en el documento. Las dos últimas
siguen mal, y **fallan por razones distintas**. Para verlo, variamos cuántos fragmentos
le pasamos al modelo (`k`):

In [ ]:
casos = [("¿Qué porcentaje de la nota es el trabajo final?", "Peso (%): 30"),
         ("¿Cuándo vemos la clase de LLMs?", "Sem.: 8;")]

filas = []
for pregunta, clave in casos:
    rango = puesto(pregunta, clave)
    for k in (1, 3, 10):
        respuesta, _ = rag(pregunta, k=k)
        filas.append({"pregunta": pregunta, "puesto del correcto": rango, "k": k,
                      "¿entra al contexto?": "sí" if rango <= k else "no",
                      "respuesta": respuesta})

pd.set_option("display.max_colwidth", 90)
pd.DataFrame(filas)

Dos maneras de fallar, y cada una pide un arreglo opuesto:

**Falla de recuperación** — «¿qué porcentaje…?». El fragmento correcto está en el puesto 9.
Con `k = 1` o `k = 3` no entra al contexto, y el modelo inventa un porcentaje con total
seguridad. Solo con `k = 10` lo recibe y responde bien. *Arreglo: mejor búsqueda o más `k`.*

**Falla de generación** — «¿cuándo vemos LLMs?». El fragmento correcto está **primero**.
Con `k = 1` responde bien. Con `k = 3` se distrae con el horario de clases, y con `k = 10`
se pierde del todo. *Arreglo: menos ruido o un modelo más grande.*

**No existe un `k` correcto para todo.** Por eso un sistema RAG se evalúa en dos pasos:
primero lo que recupera (sección 6.4), después lo que responde con eso.

## Lo que comprobamos

| Slide | Afirmación | Veredicto | Evidencia |
|---|---|---|---|
| 14 | Los tokens son IDs enteros | ✅ | Reproducimos los 32 IDs exactos (`o200k_base`) |
| 13 | Tokens ≠ palabras | ✅ | «desafortunadamente» = 4 tokens; «Lima» = 2 |
| 6 | Sale una distribución de probabilidad | ✅ | 151 936 probabilidades que suman 1 |
| 7–9 | Generación recursiva con criterio de parada | ✅ | Nuestro bucle de 10 líneas = `generate` |
| 16 | Existe $W_E$, un vector por token | ✅ | 151 936 × 896, el 28 % del modelo |
| 19 | Significado parecido, vectores parecidos | ✅ con matices | Peru → Bolivia, Lima; pero inflation → inflatable |
| 21 | rey − hombre + mujer ≈ reina | ✅ en inglés | «queen» primero; Tokio − Japón + Perú → Lima |
| 22 | Relaciones de comida y nacionalidad | ⚠️ | La dirección existe (tacos), pero «ceviche» no es un token |
| 24–26 | Signo del producto punto | ✅ | Base de todo el buscador de la sección 6 |

Y lo que las slides no dicen pero ahora sabes:

- El español **cuesta más tokens** que el inglés.
- Los **valores por defecto** del fabricante cambian las respuestas: repórtalos.
- **Temperatura 0** para extraer datos; más alta para crear.
- Sin contexto, un LLM **alucina con seguridad**.
- Un RAG falla de **dos maneras** —no recupera, o recupera y no usa— y más contexto no
  siempre ayuda.

## Ejercicios

**1 · Tu propio tokenizador.** Toma un párrafo de una noticia peruana y cuenta sus tokens
con `o200k_base` y con Qwen. ¿Cuál es más eficiente en español? Estima la diferencia de
costo para 1 millón de párrafos con un precio de tu elección.

**2 · Arreglar la recuperación.** En 6.5 hay preguntas cuyo fragmento correcto no entra al
contexto. Prueba tres arreglos y mide cada uno con la tabla de 6.4: (a) subir `k` a 5,
(b) combinar los puntajes de TF-IDF y embeddings (búsqueda híbrida), (c) cambiar cómo se
trocean las filas del cronograma. ¿Cuál mejora más el *hit@3* en parafraseo?

**3 · Temperatura y extracción.** Repite la pregunta «¿Cuánto tiempo dura el pitch?» 10 veces
con `do_sample=True` y temperatura 1.0. ¿Cuántas respuestas son correctas? Compáralo con
temperatura 0 y explica el resultado usando la sección 2.

**4 · Analogías en español.** Encuentra tres analogías que **sí** funcionen en la $W_E$ de
Qwen usando palabras en español que sean un solo token. Pista: prueba primero con
`id_unico()` para filtrar candidatas.

**5 · Otro documento.** Cambia el sílabo por el reglamento de estudiantes de tu universidad
(o cualquier PDF largo), arma 10 preguntas con su fragmento correcto y reporta *hit@3*.